# Measure grouped predictor permutation diagnostics

In [ ]:
"""Post-hoc model interpretation diagnostics.

This notebook adds feature-group permutation diagnostics for the annual, winter
and summer XGBoost models. The analysis is intended as a scientific
interpretation check: it quantifies how much model performance degrades when a
physically interpretable predictor group is permuted.
"""

from __future__ import annotations

import pickle
from pathlib import Path

import dill
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


from os import environ

configured_root = environ.get("WGMS_MB_ROOT")
ROOT = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT / "01_Dataset.ipynb").is_file():
        ROOT = ROOT.parent
    if not (ROOT / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set WGMS_MB_ROOT.")
elif not ROOT.is_dir():
    raise FileNotFoundError(f"The configured WGMS_MB_ROOT directory does not exist: {ROOT}")
OUT = ROOT / "results" / "analysis" / "group_permutation_importance.csv"
RANDOM_STATE = 42
N_REPEATS = 20

PAYLOADS = {
    "annual": ROOT / "models" / "annual" / "preprocessed_data.pkl",
    "winter": ROOT / "models" / "annual_seasonal" / "preprocessed_winter_data.pkl",
    "summer": ROOT / "models" / "annual_seasonal" / "preprocessed_summer_data.pkl",
}

MODELS = {
    "annual": ROOT / "models" / "annual" / "xgboost_annual_global.pkl",
    "winter": ROOT / "models" / "annual_seasonal" / "xgboost_winter_global.pkl",
    "summer": ROOT / "models" / "annual_seasonal" / "xgboost_summer_global.pkl",
}

GROUP_DEFINITIONS = {
    "Topography / geometry": ["static"],
    "Temperature / melt": ["t2m", "smlt", "tsn"],
    "Snow / precipitation": ["rsn", "sde", "sf", "tp"],
    "Radiation / flux / pressure": ["fal", "slhf", "ssr", "strd", "sshf", "ssrd", "sp"],
}


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = y_true[mask]
    yp = y_pred[mask]
    rmse = float(mean_squared_error(yt, yp) ** 0.5)
    return {
        "R2": float(r2_score(yt, yp)),
        "RMSE": rmse,
        "MAE": float(mean_absolute_error(yt, yp)),
        "Pearson_r": float(np.corrcoef(yt, yp)[0, 1]) if np.std(yp) > 0 else np.nan,
    }


def load_payload(path: Path) -> dict:
    with path.open("rb") as handle:
        payload = dill.load(handle)
    columns = list(payload["feature_columns"])
    return {
        "X": pd.DataFrame(np.asarray(payload["X_all"], dtype=float), columns=columns),
        "y": np.asarray(payload["y_all"], dtype=float),
        "feature_columns": columns,
        "feat_groups": payload["feat_groups"],
    }


def load_model(path: Path):
    with path.open("rb") as handle:
        obj = pickle.load(handle)
    if isinstance(obj, dict) and "model" in obj:
        return obj["model"], obj.get("scaler")
    return obj, None


def predict(model, scaler, X: pd.DataFrame) -> np.ndarray:
    if scaler is not None:
        X_values = scaler.transform(X)
    else:
        X_values = X.to_numpy(dtype=float)
    return np.asarray(model.predict(X_values), dtype=float)


def columns_for_group(feat_groups: dict, feature_columns: list[str], group_name: str) -> list[str]:
    columns: list[str] = []
    for key in GROUP_DEFINITIONS[group_name]:
        if key in feat_groups:
            columns.extend(feat_groups[key])
        else:
            columns.extend([col for col in feature_columns if col == key or col.endswith(f"_{key}")])
    if not columns:
        raise ValueError(f"No columns matched feature group {group_name}")
    return columns


def permute_columns(X: pd.DataFrame, columns: list[str], rng: np.random.Generator) -> pd.DataFrame:
    X_perm = X.copy()
    for col in columns:
        X_perm[col] = rng.permutation(X_perm[col].to_numpy())
    return X_perm


def summarize_permutation(
    target: str,
    group_name: str,
    y: np.ndarray,
    baseline_metrics: dict[str, float],
    permuted_predictions: list[np.ndarray],
    n_columns: int,
) -> dict[str, float | str | int]:
    rows = [compute_metrics(y, pred) for pred in permuted_predictions]
    df = pd.DataFrame(rows)
    return {
        "target": target,
        "feature_group": group_name,
        "n_features_permuted": n_columns,
        "n_repeats": len(permuted_predictions),
        "baseline_R2": baseline_metrics["R2"],
        "baseline_RMSE": baseline_metrics["RMSE"],
        "baseline_MAE": baseline_metrics["MAE"],
        "permuted_R2_mean": df["R2"].mean(),
        "permuted_R2_sd": df["R2"].std(ddof=1),
        "permuted_RMSE_mean": df["RMSE"].mean(),
        "permuted_RMSE_sd": df["RMSE"].std(ddof=1),
        "permuted_MAE_mean": df["MAE"].mean(),
        "permuted_MAE_sd": df["MAE"].std(ddof=1),
        "delta_RMSE_mean": df["RMSE"].mean() - baseline_metrics["RMSE"],
        "delta_RMSE_sd": df["RMSE"].std(ddof=1),
        "delta_R2_mean": df["R2"].mean() - baseline_metrics["R2"],
        "analysis": "global_model_group_permutation",
        "interpretation": "model sensitivity diagnostic, not causal attribution",
    }


def main() -> None:
    rng = np.random.default_rng(RANDOM_STATE)
    out_rows = []

    for target in ["annual", "winter", "summer"]:
        payload = load_payload(PAYLOADS[target])
        model, scaler = load_model(MODELS[target])
        X = payload["X"]
        y = payload["y"]
        baseline_pred = predict(model, scaler, X)
        baseline_metrics = compute_metrics(y, baseline_pred)

        for group_name in GROUP_DEFINITIONS:
            group_cols = columns_for_group(payload["feat_groups"], payload["feature_columns"], group_name)
            missing = sorted(set(group_cols) - set(payload["feature_columns"]))
            if missing:
                raise ValueError(f"{target} {group_name} has missing columns: {missing[:5]}")
            predictions = []
            for _ in range(N_REPEATS):
                X_perm = permute_columns(X, group_cols, rng)
                predictions.append(predict(model, scaler, X_perm))
            out_rows.append(
                summarize_permutation(
                    target=target,
                    group_name=group_name,
                    y=y,
                    baseline_metrics=baseline_metrics,
                    permuted_predictions=predictions,
                    n_columns=len(group_cols),
                )
            )

    out_df = pd.DataFrame(out_rows).sort_values(["target", "delta_RMSE_mean"], ascending=[True, False])
    OUT.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUT, index=False)
    print(f"Wrote {OUT}")


if __name__ == "__main__":
    main()